In [2]:
# Cell 1: Setup (run first after every session restart)
print("🚀 Setting up Colab environment...\n")

# 1. Verify GPU
!nvidia-smi | head -20

# 2. Install packages
print("\n📦 Installing dependencies...")
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets transformers
print("✅ Installation complete!")
print("\n⚠️  IMPORTANT: Restart runtime now, then skip this cell next time.")

🚀 Setting up Colab environment...

Mon Sep 21 17:58:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------

In [3]:
# Cell 2: Load project (run after restart)
import os
import sys

# Clone repo
if not os.path.exists("movie-content-safety"):
    !git clone https://github.com/flaviocr2012/movie-content-safety.git
%cd movie-content-safety

# Add src to path
sys.path.insert(0, os.path.abspath("src"))

# Create .env
with open(".env", "w") as f:
    f.write("GROQ_API_KEY=dummy\nGROQ_MODEL=openai/gpt-oss-20b\n")
    f.write("TMDB_API_KEY=dummy\nLANGCHAIN_TRACING_V2=false\n")

# Import config
from fine_tuning_config import PRESETS
config = PRESETS["dpo_preference"]

print("✅ Project loaded")
print(f"✅ Model: {config.training.model_name}")
print(f"✅ Method: {config.method.upper()}")

Cloning into 'movie-content-safety'...
remote: Enumerating objects: 162, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 162 (delta 80), reused 123 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (162/162), 2.03 MiB | 27.77 MiB/s, done.
Resolving deltas: 100% (80/80), done.
/content/movie-content-safety
⚠️ WARNING: LANGCHAIN_API_KEY not found — tracing disabled.
✅ Project loaded
✅ Model: unsloth/llama-3.1-8b-instruct-bnb-4bit
✅ Method: DPO


In [8]:
# Cell 2.5: Override config for T4 GPU
# T4 doesn't support bf16, so we use fp16 instead
config.training.fp16 = True
config.training.bf16 = False

print(f"✅ Config updated for T4 GPU:")
print(f"   fp16: {config.training.fp16}")
print(f"   bf16: {config.training.bf16}")

✅ Config updated for T4 GPU:
   fp16: True
   bf16: False


In [4]:
# Cell 3 (EXPANDED): Generate preference data (40+ pairs)
import json
import os

os.makedirs("data", exist_ok=True)
preference_file = "data/preference_pairs.jsonl"

# Movie data: (title, overview, genres, rating, expected_classification)
movies_data = [
    # SAFE MOVIES (20)
    ("The Lion King", "A young lion prince flees his kingdom after his father's murder.", "Animation, Adventure, Drama", "8.5", "Safe"),
    ("Toy Story", "A cowboy doll is threatened by a new spaceman figure.", "Animation, Adventure, Comedy", "8.3", "Safe"),
    ("Finding Nemo", "A clownfish searches for his lost son.", "Animation, Adventure, Comedy", "8.2", "Safe"),
    ("Frozen", "A princess with ice powers must save her kingdom.", "Animation, Adventure, Comedy", "7.4", "Safe"),
    ("Moana", "A young woman sails to save her island.", "Animation, Adventure, Comedy", "7.6", "Safe"),
    ("Coco", "A boy travels to the Land of the Dead.", "Animation, Adventure, Family", "8.4", "Safe"),
    ("Up", "An old man flies his house to South America.", "Animation, Adventure, Comedy", "8.3", "Safe"),
    ("Inside Out", "A girl's emotions come to life.", "Animation, Adventure, Comedy", "8.1", "Safe"),
    ("Zootopia", "A bunny cop teams up with a fox.", "Animation, Adventure, Comedy", "8.0", "Safe"),
    ("Shrek", "An ogre rescues a princess.", "Animation, Adventure, Comedy", "7.9", "Safe"),
    ("The Incredibles", "A family of superheroes saves the world.", "Animation, Action, Adventure", "8.0", "Safe"),
    ("Monsters, Inc.", "Two monsters befriend a human child.", "Animation, Adventure, Comedy", "8.1", "Safe"),
    ("Ratatouille", "A rat dreams of becoming a chef.", "Animation, Adventure, Comedy", "8.1", "Safe"),
    ("Wall-E", "A robot falls in love on a deserted Earth.", "Animation, Adventure, Family", "8.4", "Safe"),
    ("Aladdin", "A street urchin finds a magic lamp.", "Animation, Adventure, Comedy", "8.0", "Safe"),
    ("Beauty and the Beast", "A prince is cursed to be a beast.", "Animation, Family, Fantasy", "8.0", "Safe"),
    ("Tangled", "A princess with magic hair escapes her tower.", "Animation, Adventure, Comedy", "7.7", "Safe"),
    ("Brave", "A princess must undo a curse.", "Animation, Adventure, Comedy", "7.1", "Safe"),
    ("How to Train Your Dragon", "A Viking befriends a dragon.", "Animation, Action, Adventure", "8.1", "Safe"),
    ("Paddington", "A bear travels to London.", "Adventure, Comedy, Family", "7.2", "Safe"),
    # NOT SAFE MOVIES (20)
    ("The Conjuring", "Paranormal investigators help a terrorized family.", "Horror, Mystery, Thriller", "7.5", "Not safe"),
    ("The Shining", "A family is haunted in an isolated hotel.", "Horror, Drama", "8.4", "Not safe"),
    ("Hereditary", "A family is haunted by tragic occurrences.", "Horror, Drama, Mystery", "7.3", "Not safe"),
    ("Paranormal Activity", "A couple is disturbed by a demonic presence.", "Horror", "6.3", "Not safe"),
    ("Get Out", "A man visits his girlfriend's family.", "Horror, Mystery, Thriller", "7.8", "Not safe"),
    ("A Quiet Place", "A family hides from monsters in silence.", "Horror, Drama, Sci-Fi", "7.5", "Not safe"),
    ("The Dark Knight", "Batman faces the Joker.", "Action, Crime, Drama", "9.0", "Not safe"),
    ("Pulp Fiction", "Interconnected stories of crime.", "Crime, Drama", "8.9", "Not safe"),
    ("The Godfather", "The story of a crime family.", "Crime, Drama", "9.2", "Not safe"),
    ("The Matrix", "A hacker discovers reality is fake.", "Action, Sci-Fi", "8.7", "Not safe"),
    ("Jurassic Park", "Dinosaurs run loose in a theme park.", "Action, Adventure, Sci-Fi", "8.2", "Not safe"),
    ("Jaws", "A great white shark terrorizes a beach.", "Adventure, Thriller", "8.1", "Not safe"),
    ("Die Hard", "A cop fights terrorists in a skyscraper.", "Action, Thriller", "8.2", "Not safe"),
    ("The Terminator", "A cyborg is sent to kill a waitress.", "Action, Sci-Fi", "8.1", "Not safe"),
    ("Se7en", "Two detectives hunt a serial killer.", "Crime, Drama, Mystery", "8.6", "Not safe"),
    ("Fight Club", "An insomniac forms an underground fight club.", "Drama, Thriller", "8.8", "Not safe"),
    ("Scarface", "A Cuban immigrant becomes a drug lord.", "Crime, Drama", "8.3", "Not safe"),
    ("Joker", "A failed comedian becomes a criminal.", "Crime, Drama, Thriller", "8.4", "Not safe"),
    ("Saving Private Ryan", "Soldiers rescue a paratrooper in WWII.", "Drama, War", "8.6", "Not safe"),
    ("Schindler's List", "A businessman saves Jewish workers in WWII.", "Biography, Drama, History", "9.0", "Not safe"),
]

def build_pair(title, overview, genres, rating, expected):
    """Build a preference pair for a movie."""
    prompt = (
        f"Movie: {title}\n"
        f"Overview: {overview}\n"
        f"Genres: {genres}\n"
        f"Rating: {rating}\n\n"
        f"Is this movie appropriate for children aged 5-10?"
    )
    if expected == "Safe":
        chosen = (
            f"Classification: Safe for children\n"
            f"Explanation: {title} is a family-friendly movie with positive themes and no concerning content for children aged 5-10."
        )
        rejected = (
            f"Classification: Not safe for children\n"
            f"Explanation: {title} has some content that may not be suitable."
        )
    else:
        chosen = (
            f"Classification: Not safe for children\n"
            f"Explanation: {title} contains content that is not appropriate for children aged 5-10, including intense scenes or mature themes."
        )
        rejected = (
            f"Classification: Safe for children\n"
            f"Explanation: {title} is a popular movie that many families enjoy."
        )
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

preference_pairs = [build_pair(*m) for m in movies_data]

with open(preference_file, "w") as f:
    for pair in preference_pairs:
        f.write(json.dumps(pair) + "\n")

print(f"✅ Generated {len(preference_pairs)} preference pairs")
print(f"   - Safe: {sum(1 for m in movies_data if m[4] == 'Safe')}")
print(f"   - Not safe: {sum(1 for m in movies_data if m[4] == 'Not safe')}")
print(f"📁 Saved to: {preference_file}")

✅ Generated 40 preference pairs
   - Safe: 20
   - Not safe: 20
📁 Saved to: data/preference_pairs.jsonl


In [5]:
# Cell 4: Load base model
from unsloth import FastLanguageModel
import torch

print(f"🔍 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("❌ No GPU detected. Enable T4 GPU in Runtime settings.")

print(f"\n🔄 Loading model: {config.training.model_name}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config.training.model_name,
    max_seq_length=config.training.max_seq_length,
    load_in_4bit=config.training.load_in_4bit,
    dtype=None,
)

print(f"\n✅ Model loaded!")
print(f"📊 Total parameters: {model.num_parameters():,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🔍 CUDA available: True
🎮 GPU: Tesla T4
💾 VRAM: 15.6 GB

🔄 Loading model: unsloth/llama-3.1-8b-instruct-bnb-4bit
==((====))==  Unsloth 2026.9.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.



✅ Model loaded!
📊 Total parameters: 8,030,261,248


In [7]:
!nvidia-smi

Mon Sep 21 14:34:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
# Cell 5: Add LoRA adapters
print("🔄 Adding LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r=config.lora.r,
    target_modules=config.lora.target_modules,
    lora_alpha=config.lora.lora_alpha,
    lora_dropout=config.lora.lora_dropout,
    bias=config.lora.bias,
    use_gradient_checkpointing=config.lora.use_gradient_checkpointing,
    random_state=config.training.seed,
    use_rslora=config.lora.use_rslora,
    loftq_config=config.lora.loftq_config,
)

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\n✅ LoRA adapters added!")
print(f"📊 Trainable parameters: {trainable_params:,}")
print(f"📊 Total parameters:     {total_params:,}")
print(f"📊 Trainable %:          {100 * trainable_params / total_params:.2f}%")

🔄 Adding LoRA adapters...


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.9.7 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers. The fused LoRA kernels were skipped because lora_dropout = 0.05, which is why the counts are zero. Training is unaffected.



✅ LoRA adapters added!
📊 Trainable parameters: 41,943,040
📊 Total parameters:     4,582,543,360
📊 Trainable %:          0.92%


In [9]:
# Verification: Check model has trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

if trainable == 0:
    print("❌ ERROR: No trainable parameters!")
    print("   → You need to run Cell 5 first (LoRA adapters)")
else:
    print(f"✅ Model is ready for training")
    print(f"📊 Trainable parameters: {trainable:,}")

✅ Model is ready for training
📊 Trainable parameters: 41,943,040


In [6]:
# Cell 6 (FIXED): Format dataset with chat template
from datasets import load_dataset

# Ensure tokenizer has chat template
from unsloth import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

# Load preference pairs
dataset = load_dataset(
    "json",
    data_files=preference_file,
    split="train"
)

# Split
split = dataset.train_test_split(test_size=config.training.test_size, seed=config.training.seed)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"✅ Dataset loaded!")
print(f"📊 Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

# Format with proper chat template + EOS tokens
def format_conversation(example):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a movie content safety classifier. "
                "Your task is to determine if a movie is appropriate for children aged 5-10. "
                "Always respond with a clear classification and a brief explanation."
            )
        },
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["chosen"]},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    }

train_formatted = train_dataset.map(format_conversation)
eval_formatted = eval_dataset.map(format_conversation)

print(f"\n📝 Sample formatted prompt:\n")
print("=" * 70)
print(train_formatted[0]["text"][:600])
print("=" * 70)

Generating train split: 0 examples [00:00, ? examples/s]

✅ Dataset loaded!
📊 Train: 32 | Eval: 8


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]


📝 Sample formatted prompt:

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are a movie content safety classifier. Your task is to determine if a movie is appropriate for children aged 5-10. Always respond with a clear classification and a brief explanation.<|eot_id|><|start_header_id|>user<|end_header_id|>

Movie: A Quiet Place
Overview: A family hides from monsters in silence.
Genres: Horror, Drama, Sci-Fi
Rating: 7.5

Is this movie appropriate for children aged 5-10?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Classification: 


In [10]:
# Cell 7 (FIXED): Initialize trainer
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

# Auto-detect precision (T4 doesn't support bf16)
if not torch.cuda.is_bf16_supported():
    print("⚠️ bf16 not supported — using fp16")
    config.training.bf16 = False
    config.training.fp16 = True

training_args = TrainingArguments(**config.training.to_dict())

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    eval_dataset=eval_formatted,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=config.training.max_seq_length,
    packing=False,
)

# Train only on assistant responses (mask the user prompt)
try:
    from unsloth import train_on_responses_only
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
        response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
    )
    print("✅ Trainer initialized (training on responses only)")
except ImportError:
    print("✅ Trainer initialized (full sequence)")

print(f"📊 Training args: {training_args.num_train_epochs} epochs, batch {training_args.per_device_train_batch_size}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


⚠️ bf16 not supported — using fp16
Unsloth: We found double BOS tokens - we shall remove one automatically.
Unsloth: We found double BOS tokens - we shall remove one automatically.


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

✅ Trainer initialized (training on responses only)
📊 Training args: 2 epochs, batch 1


In [11]:
# Cell 8: Train!
print("🚀 Starting training...")
print("⏱️  This will take ~10-30 minutes on a T4 GPU")

trainer_stats = trainer.train()

print(f"\n✅ Training complete!")
print(f"📊 Final loss: {trainer_stats.training_loss:.4f}")
print(f"⏱️  Total time: {trainer_stats.metrics['train_runtime']:.0f}s")

🚀 Starting training...
⏱️  This will take ~10-30 minutes on a T4 GPU


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 32 | Num Epochs = 2 | Total steps = 16
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/

Step,Training Loss,Validation Loss
16,1.459728,0.625837


Filter:   0%|          | 0/8 [00:00<?, ? examples/s]

Unsloth: Restored added_tokens_decoder metadata in ./fine_tuned_model/checkpoint-16/tokenizer_config.json.



✅ Training complete!
📊 Final loss: 1.1732
⏱️  Total time: 82s


In [12]:
# Cell 9: Save the model
output_dir = "./fine_tuned_movie_classifier"

# Save LoRA adapters only (small, ~50MB)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ LoRA adapters saved to: {output_dir}")

# Check file size
import os
total_size = sum(
    os.path.getsize(os.path.join(output_dir, f))
    for f in os.listdir(output_dir)
    if os.path.isfile(os.path.join(output_dir, f))
)
print(f"📦 Total size: {total_size / 1e6:.1f} MB")

Unsloth: Restored added_tokens_decoder metadata in ./fine_tuned_movie_classifier/tokenizer_config.json.


✅ LoRA adapters saved to: ./fine_tuned_movie_classifier
📦 Total size: 185.1 MB


In [13]:
# Cell 10 (FIXED): Test the fine-tuned model
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

test_movies = [
    {
        "title": "The Conjuring",
        "prompt": "Movie: The Conjuring\nOverview: Paranormal investigators help a family terrorized by a dark presence.\nGenres: Horror, Mystery, Thriller\nRating: 7.5\n\nIs this movie appropriate for children aged 5-10?"
    },
    {
        "title": "Finding Nemo",
        "prompt": "Movie: Finding Nemo\nOverview: A clownfish sets out on a journey to find his son.\nGenres: Animation, Adventure, Comedy\nRating: 8.2\n\nIs this movie appropriate for children aged 5-10?"
    },
    {
        "title": "Jurassic Park",
        "prompt": "Movie: Jurassic Park\nOverview: A paleontologist must protect kids after the park's cloned dinosaurs run loose.\nGenres: Action, Adventure, Sci-Fi\nRating: 8.2\n\nIs this movie appropriate for children aged 5-10?"
    },
]

for movie in test_movies:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a movie content safety classifier. "
                "Your task is to determine if a movie is appropriate for children aged 5-10. "
                "Always respond with a clear classification and a brief explanation."
            )
        },
        {"role": "user", "content": movie["prompt"]},
    ]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([input_text], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.1,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    print(f"\n{'='*70}")
    print(f"🎬 {movie['title']}")
    print(f"📌 {response.strip()}")
    print(f"{'='*70}")

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🎬 The Conjuring
📌 Classification: Not safe for children
Explanation: The movie contains mature themes and intense scenes that are not suitable for children aged 5-10.


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🎬 Finding Nemo
📌 Classification: Safe
Explanation: Finding Nemo is a family-friendly movie with no mature themes or content that would be unsuitable for children aged 5-10.

🎬 Jurassic Park
📌 Classification: Not Safe
Explanation: The movie contains scenes with intense action, mild violence, and some frightening imagery, which may be too intense for children aged 5-10.


In [14]:
# Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
save_path = "/content/drive/MyDrive/movie-content-safety/fine_tuned_model"
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Model saved to Google Drive: {save_path}")

Mounted at /content/drive


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/movie-content-safety/fine_tuned_model/tokenizer_config.json.


✅ Model saved to Google Drive: /content/drive/MyDrive/movie-content-safety/fine_tuned_model
